# Modelagem

Dois modelos: Random Forest e XGBoost. Os dois lidam bem com dados tabulares e não precisam de muita engenharia de features. A ideia não é achar o melhor modelo do planeta, mas comparar dois candidatos sólidos e seguir com o que performar melhor neste dataset.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, roc_curve,
)
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/tracks_clean.csv")

FEATURES = [
    "danceability", "energy", "valence", "tempo",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "duration_ms",
]

X = df[FEATURES]
y = df["is_hit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")
print(f"Hits no teste: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

Treino: 65075 | Teste: 16269
Hits no teste: 1767 (10.9%)


## Random Forest

In [2]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     14502
           1       0.52      0.02      0.04      1767

    accuracy                           0.89     16269
   macro avg       0.71      0.51      0.49     16269
weighted avg       0.85      0.89      0.84     16269

ROC-AUC: 0.7005


In [3]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig = px.imshow(
    cm_rf, text_auto=True,
    labels=dict(x="Predito", y="Real"),
    x=["Não-hit", "Hit"], y=["Não-hit", "Hit"],
    title="Matriz de confusão — Random Forest",
    color_continuous_scale="Blues",
)
fig.update_layout(height=450, template="plotly_white")
fig.show()

## XGBoost

In [4]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42,
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}")

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     14502
           1       0.61      0.01      0.02      1767

    accuracy                           0.89     16269
   macro avg       0.75      0.50      0.48     16269
weighted avg       0.86      0.89      0.84     16269

ROC-AUC: 0.7148


In [5]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

fig = px.imshow(
    cm_xgb, text_auto=True,
    labels=dict(x="Predito", y="Real"),
    x=["Não-hit", "Hit"], y=["Não-hit", "Hit"],
    title="Matriz de confusão — XGBoost",
    color_continuous_scale="Reds",
)
fig.update_layout(height=450, template="plotly_white")
fig.show()

## Curva ROC comparativa

A curva ROC mostra o tradeoff entre acertar hits e errar não-hits. Quanto mais a curva se afasta da diagonal cinza, melhor o modelo está separando as classes.

In [6]:
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)

auc_rf = roc_auc_score(y_test, y_prob_rf)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr_rf, y=tpr_rf, mode="lines",
    name=f"Random Forest (AUC={auc_rf:.3f})",
))
fig.add_trace(go.Scatter(
    x=fpr_xgb, y=tpr_xgb, mode="lines",
    name=f"XGBoost (AUC={auc_xgb:.3f})",
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    line=dict(dash="dash", color="gray"), name="Aleatório",
))
fig.update_layout(
    title="Curva ROC: Random Forest vs XGBoost",
    xaxis_title="Taxa de falsos positivos",
    yaxis_title="Taxa de verdadeiros positivos",
    height=500, template="plotly_white",
)
fig.show()

## Qual modelo ganha?

Compara os números de AUC, F1 e recall dos dois. Na prática, a diferença costuma ser pequena em dados tabulares. O que pesa pra gravadora é o recall na classe hit — quantos hits reais o modelo deixa de identificar. Um modelo que perde hits potenciais custa mais do que um que eventualmente superestima algumas músicas.

Seguimos com o que tiver melhor AUC.

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "Random Forest": [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf),
        auc_rf,
    ],
    "XGBoost": [
        accuracy_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb),
        auc_xgb,
    ],
}).round(4)

metrics

,Métrica,Random Forest,XGBoost
0,Accuracy,0.8916,0.8918
1,Precision,0.5238,0.6071
2,Recall,0.0187,0.0096
3,F1,0.0361,0.0189
4,ROC-AUC,0.7005,0.7148
